# Forecasting Business and Financial Metrics — Dataset + Data Collection

**Dataset:** Quarterly financial statements of 400 US-listed companies, 2018–2026
**File:** `kvartalni_podaci (2).csv` (same folder as this notebook, or `C:\Users\User\Desktop`)

## 1. Project questions

1. **Revenue:** How will revenue change over the next four quarters?
2. **Financial stability:** Which companies are at risk of weakening financial stability (negative trailing operating cash flow *and* current ratio below 1)?
3. **Profitability:** Will operating margin improve or decline next quarter?

This notebook covers where the data comes from, whether it is suitable, the data dictionary, every cleaning decision, and a first exploratory analysis (EDA) aimed at these three questions. It is self-contained: all cleaning code is in the notebook.

## 2. Data source and usage rules

| Item | Description |
|---|---|
| Origin | Figures companies report to the US Securities and Exchange Commission (SEC) in 10-Q / 10-K filings, published as XBRL "Company Facts" through SEC EDGAR (`https://data.sec.gov/api/xbrl/companyfacts/CIK##########.json`). Column `cik` is the EDGAR company key and `revenue_concept` holds the original XBRL tag. |
| Type of data | **Public** corporate disclosures. No personal data, no confidential or proprietary records. |
| Usage rules | SEC EDGAR content is public and may be copied and redistributed; automated access must follow the SEC fair-access policy (declared User-Agent, max. 10 requests/second). |
| Unit of observation | One company × one fiscal quarter (primary key: `cik` + `period_end`). |
| Coverage | 400 US-listed companies (S&P 500 snapshot incl. Dow Jones and Nasdaq members), fiscal quarters ending Jan 2018 – Oct 2026. |
| Collected | **9 November 2026** – dataset built from SEC EDGAR Company Facts (filings available up to that date). |
| Files | Raw: `kvartalni_podaci (2).csv` (never modified). Outputs of this notebook: `capstone_output/quarterly_clean.csv` and `capstone_output/data_dictionary.csv` next to the raw file. |

Quarterly values that a company does not report directly (mostly Q4, which only appears in the annual 10-K) were derived as differences of cumulative figures – see column `net_income_method`.

## 3. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   # if missing: run  %pip install seaborn  in a separate cell
from IPython.display import display

# The CSV is looked for next to the notebook first, then at the original location
FILE_NAME = "kvartalni_podaci (2).csv"
candidates = [Path.cwd() / FILE_NAME, Path(r"C:\Users\User\Desktop") / FILE_NAME]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"'{FILE_NAME}' not found in: {[str(p.parent) for p in candidates]}")

OUT_DIR = DATA_PATH.parent / "capstone_output"
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80, "display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
print("Data file :", DATA_PATH)
print("Output dir:", OUT_DIR)

## 4. Open and inspect the raw data

In [ ]:
# utf-8-sig removes the byte-order mark at the start of the file
raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig", low_memory=False)
print(f"Rows: {raw.shape[0]:,}   Columns: {raw.shape[1]}")
raw.head()

In [ ]:
raw.dtypes.astype(str).value_counts().rename("number of columns").to_frame()

In [ ]:
# Column names with leading/trailing spaces, and the text (non-numeric) columns
print("Column names with spaces:", [c for c in raw.columns if c != c.strip()])
raw.select_dtypes(exclude="number").head(3)

### 4.1 Does the data cover what the questions need?

In [ ]:
r = raw.rename(columns=str.strip)
pe = pd.to_datetime(r["period_end"], format="%m/%d/%Y")
print("Companies:               ", r["cik"].nunique())
print("Period ends:             ", pe.min().date(), "to", pe.max().date())
print("Quarters per company:    ", r.groupby("cik").size().agg(["min", "median", "max"]).to_dict())
print("Exact duplicate rows:    ", raw.duplicated().sum())
print("Duplicate cik+period_end:", r.duplicated(["cik", "period_end"]).sum())
r.pivot_table(index="calendar_year", columns="calendar_quarter", values="cik", aggfunc="count")

**Suitability check**

- **Information:** revenue, operating income, operating expenses, current assets/liabilities, debt and operating cash flow – the inputs for all three questions – are present.
- **Time period:** nine years, 34–35 consecutive quarters for every company – a balanced panel, long enough for four-quarter-ahead targets and lagged features.
- **Geography:** US-listed SEC filers only. Conclusions do not generalise to other markets.
- **Missing dimension:** there is no sector/industry column. Question 3 (one sector) needs SIC codes, which can be added from EDGAR by `cik`.
- **Point-in-time:** the file holds final figures, not the filing date. For honest forecasting the model inputs must later be rebuilt with filing dates.

### 4.2 Data-quality problems found

In [ ]:
rev_txt = raw[" revenue "]
print("revenue stored as text, examples:", rev_txt.dropna().sample(3, random_state=1).tolist())
print("placeholder '-' instead of a value:", (rev_txt.str.strip() == "-").sum())
print("negative values in parentheses:   ", rev_txt.str.contains(r"\(", na=False).sum())
print("date format example:", raw["period_start"].iloc[0])

In [ ]:
def parse_accounting_number(s: pd.Series) -> pd.Series:
    # ' 1,234.00 ' -> 1234.0 ;  ' (5.00) ' -> -5.0 ;  ' -   ' -> NaN
    t = s.astype("string").str.strip().str.replace(",", "", regex=False)
    t = t.str.replace(r"^\((.*)\)$", r"-\1", regex=True)
    t = t.mask(t.isin(["-", ""]))
    return pd.to_numeric(t, errors="coerce").astype(float)

# Amounts that can never be negative in a quarterly statement
NON_NEGATIVE = ["revenue", "cost_of_revenue", "research_development", "capex",
                "depreciation_amortization", "dividends_paid", "share_repurchases",
                "debt_repayments", "debt_issued", "interest_paid"]

num = r.assign(revenue=parse_accounting_number(r["revenue"]))
neg = (num[NON_NEGATIVE] < 0).sum().rename("negative values").to_frame()
neg["share of Q4 rows"] = [(num.loc[num[c] < 0, "calendar_quarter"] == 4).mean() for c in NON_NEGATIVE]
neg

In [ ]:
# Scale errors: EPS and share counts
display(r.loc[r["eps_basic"].abs() > 200, ["ticker", "period_end", "eps_basic", "net_income", "shares_basic"]])
display(r.loc[r["shares_basic"] < 1e6, ["ticker", "period_end", "shares_basic", "eps_basic", "net_income"]].head(6))

In [ ]:
# calendar_year + calendar_quarter is not unique per company (52/53-week fiscal years)
dup = r[r.duplicated(["ticker", "calendar_year", "calendar_quarter"], keep=False)]
print("rows sharing ticker+calendar_year+calendar_quarter:", len(dup), "| companies:", dup["ticker"].nunique())
dup[["ticker", "period_start", "period_end", "calendar_year", "calendar_quarter"]].head(4)

In [ ]:
# Quarter length - a few companies use 12/16-week fiscal periods
days = (pe - pd.to_datetime(r["period_start"], format="%m/%d/%Y")).dt.days + 1
print(days.describe()[["min", "50%", "max"]].to_dict())
print("companies with quarters > 100 days:", sorted(r.loc[days > 100, "ticker"].unique()))

### 4.3 Missing values

In [ ]:
miss = r.isna().mean().mul(100).sort_values()
fig, ax = plt.subplots(figsize=(8, 13))
miss.plot.barh(ax=ax, color=np.where(miss > 50, "#c44e52", "#4c72b0"))
ax.set(xlabel="% missing", title="Share of missing values per column (raw data)")
plt.tight_layout(); plt.show()

In [ ]:
# Missing values are structural: they depend on the type of company, not on chance
few = num.groupby("ticker")[["revenue", "gross_profit", "inventory", "research_development"]].agg(lambda s: s.isna().mean())
print("Companies with revenue missing in >50% of quarters:", sorted(few.index[few["revenue"] > 0.5]))
print("Companies that never report gross_profit:", (few["gross_profit"] == 1).sum(), "of", len(few))
print("Companies that never report inventory:   ", (few["inventory"] == 1).sum(), "of", len(few))

Banks and insurers do not report revenue, cost of revenue or inventory under the standard tags; service companies have no inventory or R&D. A missing value therefore usually means **"not applicable / not tagged"**, not zero.

## 5. Cleaning

All changes are made by the `clean()` function below, so they are repeatable. The raw file is never modified.

| # | Problem | Decision | Why |
|---|---|---|---|
| 1 | Column name `' revenue '` has spaces; file has a BOM | Strip names, read as `utf-8-sig` | Consistent access to columns |
| 2 | `revenue` is text: thousands separators, `-` placeholders, negatives in parentheses | Parse to float; `-` → missing | Needed for any calculation |
| 3 | Dates as `M/D/YYYY` text | Convert to datetime; add `period_days` | Sorting, time series |
| 4 | Negative revenue, capex, dividends etc. | Set the value to missing, mark row in `flag_negative_fixed` | Artefacts of deriving a quarter by subtracting cumulative values; the true value is unknown, so it is not guessed |
| 5 | Share counts reported in thousands/millions, or as 0 | Rescale ×1e3 / ×1e6 when that brings the value close to the company's own median; remaining values below 1 million (zeros, ECHO 2023–24) → missing; mark in `flag_shares_fixed` | Unit error is recoverable; a 0 share count is impossible |
| 6 | EPS off by 10⁶ (Halliburton) | Divide by 1e6 where EPS ≈ 10⁶ × net income / shares | Unit error, recoverable |
| 7 | `free_cash_flow` built from negative capex | Recompute as operating cash flow − capex | Keep derived column consistent |
| 8 | `gross_profit` ≠ revenue − cost_of_revenue (>1% of revenue) | Keep values, mark in `flag_gross_profit_mismatch` | Different definitions between filers; analyst decides |
| 9 | `calendar_year`+`calendar_quarter` repeats for 52/53-week fiscal years | Keep originals, add `mid_year`/`mid_quarter` from the period midpoint; use `cik`+`period_end` as key | Fewer repeated company-quarters; joins must use the key |
| 10 | `core_available`, `balance_identity_difference` are constant | Drop | No information |
| 11 | Missing values (structural) | **Not imputed.** Each analysis uses rows where its inputs exist | Filling "not applicable" with 0 or a mean would create false data |

In [ ]:
SHARE_COLS = ["shares_basic", "shares_diluted", "shares_outstanding"]
EPS_COLS = ["eps_basic", "eps_diluted"]
CONSTANT_COLS = ["core_available", "balance_identity_difference"]


def fix_share_scale(df: pd.DataFrame, col: str) -> pd.Series:
    # Values 1,000x or 1,000,000x too small compared with the company's median are rescaled
    s = df[col]
    med = s.groupby(df["cik"]).transform("median")
    fixed = s.copy()
    for factor in (1e3, 1e6):
        ratio = s * factor / med
        hit = (s < med / 100) & ratio.between(0.5, 2)
        fixed[hit] = s[hit] * factor
    return fixed


def clean(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.rename(columns=str.strip).copy()

    # 2. revenue text -> number
    df["revenue"] = parse_accounting_number(df["revenue"])

    # 3. dates
    for c in ["period_start", "period_end"]:
        df[c] = pd.to_datetime(df[c], format="%m/%d/%Y")
    df["period_days"] = (df["period_end"] - df["period_start"]).dt.days + 1

    # 4. impossible negatives -> missing + flag
    neg = df[NON_NEGATIVE] < 0
    df["flag_negative_fixed"] = neg.any(axis=1)
    df[NON_NEGATIVE] = df[NON_NEGATIVE].mask(neg)

    # 5. share counts in wrong units: rescale if possible, otherwise (zeros, other errors) -> missing
    df["flag_shares_fixed"] = False
    for c in SHARE_COLS:
        fixed = fix_share_scale(df, c)
        fixed = fixed.mask(fixed < 1e6)
        df["flag_shares_fixed"] |= fixed.ne(df[c]) & df[c].notna()
        df[c] = fixed

    # 6. EPS off by 1e6
    implied = df["net_income"] / df["shares_basic"]
    for c in EPS_COLS:
        bad = (df[c].abs() > 1000) & ((df[c] / 1e6) / implied).between(0.5, 2)
        df.loc[bad, c] = df.loc[bad, c] / 1e6

    # 7. free cash flow consistent with cleaned capex
    df["free_cash_flow"] = df["operating_cash_flow"] - df["capex"]

    # 8. gross profit check (flag only)
    gap = (df["gross_profit"] - (df["revenue"] - df["cost_of_revenue"])).abs()
    df["flag_gross_profit_mismatch"] = (gap > 0.01 * df["revenue"].abs()).fillna(False)

    # 9. calendar quarter from the middle of the fiscal period
    mid = df["period_start"] + (df["period_end"] - df["period_start"]) / 2
    df["mid_year"] = mid.dt.year
    df["mid_quarter"] = mid.dt.quarter

    # 10. constant columns
    df = df.drop(columns=CONSTANT_COLS)

    for c in ["ticker", "company", "revenue_concept", "net_income_method", "balance_identity_status"]:
        df[c] = df[c].astype("string")
    return df.sort_values(["cik", "period_end"]).reset_index(drop=True)


df = clean(raw)
print(f"Clean data: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
checks = {
    "rows unchanged": len(df) == len(raw),
    "primary key cik+period_end unique": not df.duplicated(["cik", "period_end"]).any(),
    "no negative values in NON_NEGATIVE columns": not (df[NON_NEGATIVE] < 0).any().any(),
    "assets = liabilities_and_equity": bool(((df["assets"] - df["liabilities_and_equity"]).abs().fillna(0) < 1).all()),
    "max |EPS| plausible (< 200)": df["eps_basic"].abs().max() < 200,
    "min share count > 1 million": df[SHARE_COLS].min().min() > 1e6,
}
display(pd.Series(checks, name="passed").to_frame())
print("rows with a negative value set to missing:", int(df["flag_negative_fixed"].sum()))
print("rows with share counts fixed:             ", int(df["flag_shares_fixed"].sum()),
      "| companies:", df.loc[df["flag_shares_fixed"], "ticker"].nunique())
print("rows with gross-profit mismatch flag:     ", int(df["flag_gross_profit_mismatch"].sum()))
print("duplicate ticker+mid_year+mid_quarter:    ", int(df.duplicated(["ticker", "mid_year", "mid_quarter"]).sum()),
      "(was", int(df.duplicated(["ticker", "calendar_year", "calendar_quarter"]).sum()), "with calendar_* columns)")

In [ ]:
clean_path = OUT_DIR / "quarterly_clean.csv"
df.to_csv(clean_path, index=False, date_format="%Y-%m-%d")
print("Saved:", clean_path)

## 6. Data dictionary

All amounts are in **USD** (EPS and dividend per share in USD per share; share counts in shares). Flow items (income statement, cash flow) are for the quarter; balance-sheet items are at `period_end`. `type` describes the cleaned file. The table is also saved as `data_dictionary.csv`.

In [ ]:
DESCRIPTIONS = {
    # identifiers and time
    "cik": "SEC Central Index Key - EDGAR company identifier",
    "ticker": "Stock ticker symbol",
    "company": "Company name",
    "period_start": "First day of the fiscal quarter",
    "period_end": "Last day of the fiscal quarter (with cik: primary key)",
    "calendar_year": "Calendar year assigned in the source file",
    "calendar_quarter": "Calendar quarter (1-4) assigned in the source file",
    # income statement
    "revenue": "Total revenue for the quarter",
    "cost_of_revenue": "Cost of goods sold / cost of revenue",
    "gross_profit": "Gross profit as reported",
    "operating_income": "Operating income (loss)",
    "net_income": "Net income attributable to the company",
    "pretax_income": "Income before income taxes",
    "income_tax_expense": "Income tax expense (benefit)",
    "research_development": "Research and development expense",
    "selling_general_admin": "Selling, general and administrative expense",
    "operating_expenses": "Total operating expenses (when reported)",
    "interest_expense": "Interest expense",
    "eps_basic": "Basic earnings per share",
    "eps_diluted": "Diluted earnings per share",
    "shares_basic": "Weighted average basic shares",
    "shares_diluted": "Weighted average diluted shares",
    # balance sheet
    "assets": "Total assets",
    "current_assets": "Current assets",
    "noncurrent_assets": "Non-current assets (rarely tagged)",
    "liabilities": "Total liabilities",
    "current_liabilities": "Current liabilities",
    "noncurrent_liabilities": "Non-current liabilities (rarely tagged)",
    "equity": "Stockholders' equity attributable to the company",
    "equity_including_minority": "Equity including non-controlling interests",
    "liabilities_and_equity": "Total liabilities and equity (= assets)",
    "cash": "Cash and cash equivalents",
    "cash_restricted": "Cash, equivalents and restricted cash",
    "receivables": "Accounts receivable, net",
    "inventory": "Inventory, net",
    "ppe_net": "Property, plant and equipment, net",
    "goodwill": "Goodwill",
    "intangibles_net": "Intangible assets excl. goodwill, net",
    "accounts_payable": "Accounts payable",
    "retained_earnings": "Retained earnings (accumulated deficit)",
    "treasury_stock": "Treasury stock value",
    "short_term_borrowings": "Short-term borrowings",
    "long_term_debt_current": "Current portion of long-term debt",
    "long_term_debt_noncurrent": "Long-term debt, non-current",
    "deferred_revenue_current": "Deferred revenue / contract liabilities, current",
    # cash flow
    "operating_cash_flow": "Net cash from operating activities",
    "investing_cash_flow": "Net cash from investing activities",
    "financing_cash_flow": "Net cash from financing activities",
    "capex": "Capital expenditures (payments for PP&E)",
    "acquisitions_cash": "Cash paid for acquisitions",
    "depreciation_amortization": "Depreciation and amortization",
    "stock_compensation": "Share-based compensation",
    "dividends_paid": "Dividends paid",
    "share_repurchases": "Payments for share repurchases",
    "debt_repayments": "Repayments of debt",
    "debt_issued": "Proceeds from issuing debt",
    "income_taxes_paid": "Income taxes paid",
    "interest_paid": "Interest paid",
    "dividend_per_share": "Dividends declared per share",
    "shares_outstanding": "Shares outstanding (cover page / balance sheet)",
    # source metadata
    "revenue_concept": "XBRL tag used for revenue",
    "net_income_method": "How the quarterly figure was obtained: reported, or difference of cumulative values",
    "balance_identity_status": "Whether assets = liabilities + equity check passed",
    "free_cash_flow": "Operating cash flow - capex (recomputed in cleaning)",
    # added in cleaning
    "period_days": "Length of the fiscal quarter in days",
    "flag_negative_fixed": "True if an impossible negative value in this row was set to missing",
    "flag_shares_fixed": "True if a share count in this row was rescaled (x1e3 / x1e6) or set to missing",
    "flag_gross_profit_mismatch": "True if gross_profit differs from revenue - cost_of_revenue by >1% of revenue",
    "mid_year": "Calendar year of the period midpoint",
    "mid_quarter": "Calendar quarter of the period midpoint",
}
GROUP = (["id/time"] * 7 + ["income statement"] * 15 + ["balance sheet"] * 23
         + ["cash flow"] * 15 + ["source metadata"] * 3 + ["cash flow"] + ["added"] * 6)

raw_missing = r.isna().mean().mul(100)
dd = pd.DataFrame({
    "column": list(DESCRIPTIONS),
    "group": GROUP,
    "type": [str(df[c].dtype) for c in DESCRIPTIONS],
    "description": list(DESCRIPTIONS.values()),
    "missing_raw_%": [round(raw_missing.get(c, np.nan), 1) for c in DESCRIPTIONS],
    "missing_clean_%": [round(df[c].isna().mean() * 100, 1) for c in DESCRIPTIONS],
})
assert set(dd["column"]) == set(df.columns), set(df.columns) ^ set(dd["column"])
dd.to_csv(OUT_DIR / "data_dictionary.csv", index=False)
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 90):
    display(dd)

## 7. Exploratory data analysis

Ratios used below (added only in this notebook, not saved to the clean file):

- `operating_margin` = operating_income / revenue
- `net_margin` = net_income / revenue
- `fcf_margin` = free_cash_flow / revenue
- `current_ratio` = current_assets / current_liabilities
- `leverage` = (assets − equity) / assets
- `revenue_yoy` = revenue / revenue of the same company four quarters earlier − 1

In [ ]:
e = df.copy()
rev = e["revenue"].where(e["revenue"] > 0)
e["operating_margin"] = e["operating_income"] / rev
e["net_margin"] = e["net_income"] / rev
e["fcf_margin"] = e["free_cash_flow"] / rev
e["current_ratio"] = e["current_assets"] / e["current_liabilities"]
e["leverage"] = (e["assets"] - e["equity"]) / e["assets"]
e["revenue_yoy"] = rev / rev.groupby(e["cik"]).shift(4) - 1     # rows are sorted by cik, period_end
e["quarter"] = pd.PeriodIndex.from_fields(year=e["mid_year"], quarter=e["mid_quarter"], freq="Q")

core = ["revenue", "operating_income", "net_income", "assets", "equity", "operating_cash_flow", "free_cash_flow"]
(e[core] / 1e9).describe().T.round(2)   # USD billions

In [ ]:
ratios = ["revenue_yoy", "operating_margin", "net_margin", "fcf_margin", "current_ratio", "leverage", "eps_basic"]
e[ratios].describe(percentiles=[.01, .25, .5, .75, .99]).T

Amounts and ratios are strongly skewed (a few very large companies, a few tiny denominators), so log scales, clipping for plots and medians are used rather than means.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(np.log10(rev.dropna()), bins=50, ax=ax[0])
ax[0].set(title="Quarterly revenue (log10 USD)", xlabel="log10(revenue)")
sns.histplot(e["revenue_yoy"].clip(-0.5, 1).dropna(), bins=60, ax=ax[1])
ax[1].axvline(e["revenue_yoy"].median(), color="k", ls="--", lw=1)
ax[1].set(title="Revenue YoY growth (clipped -50%..100%)", xlabel="growth vs same quarter last year")
sns.histplot(e["operating_margin"].clip(-0.5, 0.75).dropna(), bins=60, ax=ax[2])
ax[2].axvline(e["operating_margin"].median(), color="k", ls="--", lw=1)
ax[2].set(title="Operating margin (clipped -50%..75%)", xlabel="operating income / revenue")
plt.tight_layout(); plt.show()

### 7.1 Revenue over time (Question 1)

In [ ]:
full = e[(e["quarter"] >= "2018Q1") & (e["quarter"] <= "2026Q2")]   # first and last quarters are incomplete
ts = full.groupby("quarter").agg(companies=("cik", "nunique"),
                                 total_revenue=("revenue", "sum"),
                                 total_net_income=("net_income", "sum"),
                                 median_revenue_yoy=("revenue_yoy", "median"),
                                 median_operating_margin=("operating_margin", "median"),
                                 median_net_margin=("net_margin", "median"),
                                 median_fcf_margin=("fcf_margin", "median"))
x = ts.index.to_timestamp()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(x, ts["total_revenue"] / 1e12, label="Revenue")
ax[0].plot(x, ts["total_net_income"] / 1e12, label="Net income")
ax[0].set(title="All companies combined, per quarter", ylabel="USD trillions"); ax[0].legend()
ax[1].plot(x, ts["median_revenue_yoy"] * 100)
ax[1].axhline(0, color="k", lw=.8)
ax[1].set(title="Median revenue growth vs same quarter last year", ylabel="%")
plt.tight_layout(); plt.show()
ts.loc[["2018Q1", "2020Q2", "2022Q2", "2024Q2", "2026Q2"]]

In [ ]:
# Seasonality: each quarter's revenue relative to that company's average quarter in the same year
yr = full[full["mid_year"].between(2018, 2025)].copy()
yr["rel_revenue"] = yr["revenue"] / yr.groupby(["cik", "mid_year"])["revenue"].transform("mean")
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=yr, x="mid_quarter", y="rel_revenue", showfliers=False, ax=ax)
ax.axhline(1, color="k", lw=.8)
ax.set(title="Revenue seasonality", xlabel="calendar quarter", ylabel="quarter / yearly average")
plt.tight_layout(); plt.show()
yr.groupby("mid_quarter")["rel_revenue"].median().round(3)

In [ ]:
# Does last year's growth say anything about next year's? (simple persistence check)
g = e[["cik", "revenue_yoy"]].copy()
g["revenue_yoy_next4"] = g.groupby("cik")["revenue_yoy"].shift(-4)
g = g.dropna()
g = g[g["revenue_yoy"].between(-0.5, 1) & g["revenue_yoy_next4"].between(-0.5, 1)]
rho = g[["revenue_yoy", "revenue_yoy_next4"]].corr(method="spearman").iloc[0, 1]
print("Spearman correlation, YoY growth now vs four quarters later:", round(rho, 3), f"(n={len(g):,})")

### 7.2 Financial stability (Question 2)

In [ ]:
# Strain = trailing four-quarter operating cash flow < 0 AND current ratio < 1
e["ocf_ttm"] = e.groupby("cik")["operating_cash_flow"].transform(lambda s: s.rolling(4, min_periods=4).sum())
e["strain"] = (e["ocf_ttm"] < 0) & (e["current_ratio"] < 1)
has_inputs = e["ocf_ttm"].notna() & e["current_ratio"].notna()
print("rows with both inputs:      ", f"{has_inputs.sum():,} of {len(e):,}")
print("rows in strain:             ", int(e["strain"].sum()), f"({e.loc[has_inputs, 'strain'].mean():.2%} of rows with inputs)")
print("companies ever in strain:   ", e.loc[e["strain"], "ticker"].nunique())
print("current ratio < 1 only:     ", f"{(e.loc[has_inputs, 'current_ratio'] < 1).mean():.1%} of rows")
print("negative TTM OCF only:      ", f"{(e.loc[has_inputs, 'ocf_ttm'] < 0).mean():.1%} of rows")
e.loc[e["strain"]].groupby("ticker").size().sort_values(ascending=False).head(10).rename("quarters in strain").to_frame()

The strain outcome is **rare**, so Question 2 needs care: check how many positive cases remain after the chronological split, or relax the definition (e.g. each condition separately).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
s = e[has_inputs].sample(4000, random_state=0)
ax.scatter(s["current_ratio"].clip(0, 4), (s["ocf_ttm"] / s["assets"]).clip(-0.3, 0.5), s=4, alpha=.4)
ax.axvline(1, color="#c44e52", lw=1); ax.axhline(0, color="#c44e52", lw=1)
ax.set(title="Liquidity vs cash generation (sample of 4,000 rows)",
       xlabel="current ratio (clipped at 4)", ylabel="TTM operating cash flow / assets")
plt.tight_layout(); plt.show()

### 7.3 Operating profitability (Question 3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for c in ["median_operating_margin", "median_net_margin", "median_fcf_margin"]:
    ax.plot(x, ts[c] * 100, label=c.replace("median_", "").replace("_", " "))
ax.set(title="Median margins per quarter", ylabel="%"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Baseline for Question 3: margin in the same quarter last year
m = e[["cik", "operating_margin"]].copy()
m["margin_last_year"] = m.groupby("cik")["operating_margin"].shift(4)
m["margin_last_quarter"] = m.groupby("cik")["operating_margin"].shift(1)
m = m.dropna()
m = m[m[["operating_margin", "margin_last_year", "margin_last_quarter"]].abs().lt(1).all(axis=1)]
print(f"rows usable: {len(m):,}")
print("MAE, same quarter last year (pp):", round((m["operating_margin"] - m["margin_last_year"]).abs().mean() * 100, 2))
print("MAE, previous quarter (pp):      ", round((m["operating_margin"] - m["margin_last_quarter"]).abs().mean() * 100, 2))

In [ ]:
# Relationships between key metrics (Spearman: robust to skew and outliers)
cols = ["revenue", "assets", "revenue_yoy", "operating_margin", "net_margin", "fcf_margin",
        "current_ratio", "leverage", "research_development", "stock_compensation", "capex"]
corr = e[cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Spearman correlation"); plt.tight_layout(); plt.show()

### 7.4 Largest companies and source quality

In [ ]:
last4 = e.sort_values("period_end").groupby("ticker").tail(4)
ttm = last4.groupby(["ticker", "company"]).agg(revenue_ttm=("revenue", lambda s: s.sum(min_count=4)),
                                               net_income_ttm=("net_income", "sum")).dropna()
ttm["net_margin_ttm"] = ttm["net_income_ttm"] / ttm["revenue_ttm"]
top = ttm.nlargest(15, "revenue_ttm")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index.get_level_values("ticker")[::-1], top["revenue_ttm"][::-1] / 1e9)
ax.set(title="Top 15 companies by trailing-four-quarter revenue", xlabel="USD billions")
plt.tight_layout(); plt.show()
top.assign(revenue_ttm=top["revenue_ttm"] / 1e9, net_income_ttm=top["net_income_ttm"] / 1e9).round(3)

In [ ]:
# Does data quality differ between reported and derived quarters?
e.groupby("net_income_method").agg(rows=("cik", "size"),
                                   share_q4=("calendar_quarter", lambda s: (s == 4).mean()),
                                   negative_fixed=("flag_negative_fixed", "mean"),
                                   revenue_missing=("revenue", lambda s: s.isna().mean()))

## 8. Summary

**What the data contains.** A balanced panel of 400 US-listed companies with 34–35 fiscal quarters each (13,966 rows), covering income statement, balance sheet and cash-flow items in USD, taken from public SEC filings.

**Is it suitable?** Yes, for all three questions. Revenue, net income, assets and cash flows are almost complete; revenue is missing for about 3% of rows, mostly banks. Inputs for the stability and margin questions (current assets/liabilities, operating income) are missing for 16–22% of rows, mostly financial companies, where these measures do not apply anyway.

**Main limitations**
1. Many detail columns are missing for more than half of the rows because they do not apply to every business model – analyses on those columns describe a subset of companies.
2. Derived quarters (mostly Q4) are less reliable than reported ones; rows touched by cleaning are flagged.
3. No sector column; banks and insurers are not comparable to industrial companies on margin and liquidity measures.
4. The financial-strain outcome (Question 2) is rare – class imbalance must be handled.
5. Only companies in today's list are included (survivorship bias), only US filers, and 2026 is incomplete.
6. Figures are final, not as first published – forecasting features must later be rebuilt with filing dates.

**Next steps.** Add sector (SIC code) by `cik`, exclude financial companies where measures don't apply, build lagged features, and split chronologically with a held-out final test period, comparing every model against its naive baseline.